# Qwen2.5-1.5B 예비 양자화 실험

이 노트북은 교수님 코멘트에 따라 최종 주제를 1번과 3번 중에서 고르기 위한 **예비 실험**입니다.

비교군은 FP16, RTN 4bit, 영어 위키 보정 GPTQ 4bit, 한국어 위키 보정 GPTQ 4bit입니다. 한국어와 영어 평가 문서를 각각 세 묶음으로 나누고, 각 언어에서 FP16 대비 Perplexity 저하율을 계산합니다. 원점수 자체는 언어 간에 직접 비교하지 않습니다.

예비 실험에서 확인할 내용:

1. 4bit 양자화의 한국어 저하율과 영어 저하율이 각 조건 및 평가 묶음에서 어떤 양상을 보이는가?
2. 영어 보정과 한국어 보정 GPTQ가 한국어 평가에서 차이를 만드는가?
3. 결과가 세 평가 묶음에서 일관적인가, 특정 묶음에만 나타나는가?

결과를 본 뒤에 1번(언어별 저하 분석)과 3번(한국어 품질 개선 실험) 중 하나를 정합니다. 이 노트북은 그 선택 기준을 미리 고정하지 않습니다.

**주의:** WikiText-2 영어 test와 한국어 Wikipedia는 서로 다른 평가 말뭉치입니다. 따라서 영어와 한국어의 절대 Perplexity를 비교하지 않고, 각 말뭉치 안에서 FP16 대비 상대 변화만 비교합니다. Instruct 모델에 일반 문서를 넣어 계산한 PPL은 다음 토큰 예측의 비교 지표이며, 대화 응답 품질을 직접 뜻하지 않습니다.


## 0. Colab 준비

Colab에서 런타임 유형을 **GPU(T4)** 로 설정하세요. 설치 셀은 현재 런타임에 GPTQModel이 설치되고 import되는지 확인합니다. 런타임을 새로 연결했거나 `ModuleNotFoundError`가 나면 설치 셀을 다시 실행하세요. Torch나 Transformers가 교체되어 런타임 재시작을 했다면, 재시작 후에도 설치 셀부터 다시 실행해 설치 여부를 확인합니다.

GPTQModel 설치가 Colab의 기본 Torch와 충돌하면 설치 로그와 GPU 종류를 기록해 주세요. 다른 패키지 조합으로 임의 변경하면 실험 환경이 달라질 수 있습니다.

Drive 연결 셀에서 저장 위치를 확인하세요. 연결에 실패하면 Colab 임시 저장소를 사용하며 표본과 결과 ZIP을 브라우저로 다운로드합니다. 다운로드가 실제로 완료됐는지 확인한 뒤 런타임을 종료하세요. 모델 체크포인트는 자동 다운로드하지 않습니다.


In [3]:
%pip install -q "gptqmodel==7.10.0" "datasets>=5.1.0"


In [4]:
import gc
import json
import math
import random
import sys
from pathlib import Path

import torch
import pandas as pd
import gptqmodel
import transformers
import datasets
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from gptqmodel import BACKEND, GPTQModel, GPTQConfig, RTNConfig

assert torch.cuda.is_available(), "Colab 런타임을 GPU로 바꾸고 다시 실행하세요."
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("GPTQModel:", gptqmodel.__version__)
print("Datasets:", datasets.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))



INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


W1116 07:10:24.155000 3678 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1116 07:10:24.217000 3678 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO    ┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐ │ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │ └─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘ GPT-QModel   : 7.10.0 Transformers : 5.22.1 Torch        : 2.14.1+cu130 Triton       : 3.8.0
Python: 3.13.16
Torch: 2.14.1+cu130 CUDA: 13.0
Transformers: 5.22.1
GPTQModel: 7.10.0
Datasets: 5.2.0
GPU: Tesla T4
Compute capability: (7, 5)


In [ ]:
from google.colab import drive, files
from tempfile import TemporaryDirectory

OUTPUT_DIR = None
DRIVE_AVAILABLE = False
try:
    drive.mount("/content/drive")
    if not Path("/content/drive").is_mount():
        raise RuntimeError("Google Drive 마운트를 확인할 수 없습니다.")
    OUTPUT_DIR = Path("/content/drive/MyDrive/Team22/pilot_qwen15b")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=OUTPUT_DIR) as check_dir:
        check_file = Path(check_dir) / "write_check.txt"
        check_file.write_text("ok", encoding="utf-8")
        if check_file.read_text(encoding="utf-8") != "ok":
            raise RuntimeError("Google Drive 파일 읽기/쓰기 확인에 실패했습니다.")
    DRIVE_AVAILABLE = True
    print("Google Drive 저장 확인:", OUTPUT_DIR)
except Exception as exc:
    OUTPUT_DIR = Path("/content/Team22/pilot_qwen15b")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print("Google Drive 연결 실패:", exc)
    print("표본과 결과를 브라우저로 다운로드합니다. 런타임 종료 전 다운로드 완료를 확인하세요.")


## 1. 실험 설정

평가 표본은 언어별 세 묶음, 각 묶음은 16개의 512-token 구간으로 구성합니다. GPTQ 보정 표본은 영어와 한국어 각각 128개, 최대 512 tokens입니다.

한국어 보정 표본과 한국어 평가 문서는 같은 Wikipedia 스냅샷에서 가져오되, 보정에 사용한 문서 ID는 평가 표본에서 제외합니다. 영어 보정은 영어 Wikipedia에서, 영어 평가는 WikiText-2 test에서 가져옵니다.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

WIKI_DATASET = "wikimedia/wikipedia"
WIKI_REVISION = "ad5752b5e625abfcdeefe5ae0ad2c3721c4b2619"
KOREAN_WIKI_CONFIG = "20231101.ko"
ENGLISH_WIKI_CONFIG = "20231101.en"
WIKITEXT_DATASET = "Salesforce/wikitext"
WIKITEXT_CONFIG = "wikitext-2-raw-v1"

SEED = 22
N_CALIBRATION = 128
CALIBRATION_LENGTH = 512
N_GROUPS = 3
SEQUENCES_PER_GROUP = 16
EVAL_SEQUENCE_LENGTH = 512
EVAL_TOKENS_PER_GROUP = SEQUENCES_PER_GROUP * EVAL_SEQUENCE_LENGTH

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

with open(OUTPUT_DIR / "runtime.json", "w", encoding="utf-8") as f:
    json.dump({"model_id": MODEL_ID, "wiki_revision": WIKI_REVISION, "seed": SEED,
               "torch": torch.__version__, "transformers": transformers.__version__,
               "gptqmodel": gptqmodel.__version__, "gpu": torch.cuda.get_device_name(0)},
              f, ensure_ascii=False, indent=2)


OUTPUT_DIR


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
print("Tokenizer:", tokenizer.name_or_path)
print("EOS token:", tokenizer.eos_token, tokenizer.eos_token_id)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer: Qwen/Qwen2.5-1.5B-Instruct
EOS token: <|im_end|> 151645


## 2. 고정된 보정·평가 표본 만들기

위키 문서를 seed로 섞어 보정 표본을 만듭니다. 보정에 쓴 문서 ID를 기억해 한국어 평가 표본에서 제외합니다. 각 평가 묶음은 서로 다른 문서에서 가져오고 토큰 수를 같게 맞춥니다. 표본 생성이 끝나면 텍스트를 JSON으로 저장해 런타임이 끊겨도 동일한 표본을 다시 쓸 수 있습니다.


In [8]:
def get_text(row):
    text = row.get("text", "")
    return text.strip() if isinstance(text, str) else ""

def token_count(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def load_wikipedia_stream(config, seed):
    ds = load_dataset(
        WIKI_DATASET, config, split="train", streaming=True, revision=WIKI_REVISION
    )
    return ds.shuffle(seed=seed, buffer_size=10_000)

def collect_calibration(lang, config):
    ds = load_wikipedia_stream(config, SEED + (0 if lang == "en" else 1))
    texts, ids = [], set()
    for row in ds:
        text = get_text(row)
        if not text:
            continue
        encoded = tokenizer(
            text, add_special_tokens=True, truncation=True,
            max_length=CALIBRATION_LENGTH, return_tensors="pt"
        )
        if encoded["input_ids"].shape[1] != CALIBRATION_LENGTH:
            continue
        texts.append(text)
        ids.add(str(row.get("id", "")))
        if len(texts) == N_CALIBRATION:
            break
    if len(texts) != N_CALIBRATION:
        raise RuntimeError(f"{lang}: 보정 문서를 {N_CALIBRATION}개 모으지 못했습니다.")
    return texts, ids

calib_en_texts, calib_en_ids = collect_calibration("en", ENGLISH_WIKI_CONFIG)
calib_ko_texts, calib_ko_ids = collect_calibration("ko", KOREAN_WIKI_CONFIG)
print("Calibration documents:", len(calib_en_texts), len(calib_ko_texts))


README.md:   0%|          | 0.00/129k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Calibration documents: 128 128


In [9]:
def collect_korean_eval_groups(excluded_ids):
    ds = load_wikipedia_stream(KOREAN_WIKI_CONFIG, SEED + 100)
    groups = [[] for _ in range(N_GROUPS)]
    lengths = [0] * N_GROUPS
    group_index = 0
    for row in ds:
        if str(row.get("id", "")) in excluded_ids:
            continue
        text = get_text(row)
        if not text:
            continue
        groups[group_index].append(text)
        lengths[group_index] += token_count(text)
        if lengths[group_index] >= EVAL_TOKENS_PER_GROUP:
            if group_index == N_GROUPS - 1:
                break
            group_index += 1
    if any(length < EVAL_TOKENS_PER_GROUP for length in lengths):
        raise RuntimeError(f"한국어 평가 문서를 충분히 모으지 못했습니다: {lengths}")
    return groups

def collect_english_eval_groups():
    ds = load_dataset(WIKITEXT_DATASET, WIKITEXT_CONFIG, split="test")
    groups = [[] for _ in range(N_GROUPS)]
    for index, row in enumerate(ds):
        text = get_text(row)
        if text:
            groups[index % N_GROUPS].append(text)
    return groups

raw_eval_groups = {
    "ko": collect_korean_eval_groups(calib_ko_ids),
    "en": collect_english_eval_groups(),
}

def make_token_groups(text_groups):
    token_groups = []
    for texts in text_groups:
        ids = []
        for text in texts:
            ids.extend(tokenizer(text, add_special_tokens=False)["input_ids"])
            if tokenizer.eos_token_id is not None:
                ids.append(tokenizer.eos_token_id)
            if len(ids) >= EVAL_TOKENS_PER_GROUP:
                break
        ids = ids[:EVAL_TOKENS_PER_GROUP]
        if len(ids) < EVAL_TOKENS_PER_GROUP:
            raise RuntimeError("평가 묶음 토큰 수가 부족합니다.")
        token_groups.append(ids)
    return token_groups

eval_token_groups = {
    lang: make_token_groups(groups) for lang, groups in raw_eval_groups.items()
}

with open(OUTPUT_DIR / "pilot_samples.json", "w", encoding="utf-8") as f:
    json.dump(
        {"calibration_en": calib_en_texts, "calibration_ko": calib_ko_texts,
         "eval_groups": raw_eval_groups},
        f, ensure_ascii=False
    )
if not DRIVE_AVAILABLE:
    files.download(str(OUTPUT_DIR / "pilot_samples.json"))

print({lang: [len(group) for group in groups]
       for lang, groups in eval_token_groups.items()})


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/validation-00000-of-00(…): reconstructing file:   0%|          |  0.00B /  657kB            

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

{'ko': [8192, 8192, 8192], 'en': [8192, 8192, 8192]}


## 3. Perplexity 계산

각 문서는 512-token 구간으로 평가합니다. 구간별 평균 loss를 기록하고 `exp(loss)`로 Perplexity를 계산합니다. 세 묶음은 서로 다른 문서 집합이며, 모두 모델 조건 간에 동일하게 사용됩니다.


In [10]:
def evaluate_ppl(model, token_groups, label):
    model.eval()
    rows = []
    device = torch.device("cuda:0")
    for lang, groups in token_groups.items():
        for group_idx, token_ids in enumerate(groups, start=1):
            total_nll = 0.0
            total_predicted_tokens = 0
            for start in range(0, len(token_ids), EVAL_SEQUENCE_LENGTH):
                chunk = token_ids[start:start + EVAL_SEQUENCE_LENGTH]
                if len(chunk) != EVAL_SEQUENCE_LENGTH:
                    continue
                input_ids = torch.tensor([chunk], dtype=torch.long, device=device)
                with torch.inference_mode():
                    output = model(input_ids=input_ids, labels=input_ids)
                n_predicted = input_ids.shape[1] - 1
                total_nll += float(output.loss.detach().float().cpu()) * n_predicted
                total_predicted_tokens += n_predicted
                del input_ids, output
            mean_nll = total_nll / total_predicted_tokens
            rows.append({
                "condition": label, "language": lang, "group": group_idx,
                "mean_nll": mean_nll, "ppl": math.exp(mean_nll),
                "predicted_tokens": total_predicted_tokens,
            })
    return rows

def clear_gpu_cache():
    gc.collect()
    torch.cuda.empty_cache()


In [11]:
# FP16 기준선: 세 묶음의 PPL을 먼저 계산합니다.
fp16_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16
).to("cuda:0")
fp16_rows = evaluate_ppl(fp16_model, eval_token_groups, "FP16")
del fp16_model
clear_gpu_cache()

results = list(fp16_rows)
pd.DataFrame(results)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

,condition,language,group,mean_nll,ppl,predicted_tokens
0,FP16,ko,1,2.248003,9.468811,8176
1,FP16,ko,2,2.122676,8.353463,8176
2,FP16,ko,3,2.382951,10.836835,8176
3,FP16,en,1,2.556432,12.889740,8176
4,FP16,en,2,2.632271,13.905312,8176
5,FP16,en,3,2.679174,14.573055,8176


## 4. RTN 4bit 생성 및 평가

RTN은 보정 데이터 없이 nearest rounding 방식으로 양자화합니다. 새 모델 인스턴스에서 한 번만 양자화하고, 저장한 체크포인트를 불러와 평가합니다.


In [31]:
import gptqmodel

pool_globals = gptqmodel.get_device_thread_pool.__globals__

# CPU 워커 수를 2개로 고정
pool_globals["_default_cpu_workers"] = lambda: 2

print("CPU workers:", pool_globals["_default_cpu_workers"]())

CPU workers: 2


In [38]:
def quantize_and_save(condition, quant_config, calibration_texts=None, backend=None):
    save_dir = OUTPUT_DIR / condition.lower().replace(" ", "_")
    weights_exist = any(save_dir.glob("*.safetensors")) or any(save_dir.glob("pytorch_model*.bin"))
    if (save_dir / "config.json").exists() and weights_exist:
        print(f"기존 체크포인트 사용: {save_dir}")
        return save_dir
    if save_dir.exists() and any(save_dir.iterdir()):
        raise RuntimeError(f"불완전한 기존 폴더입니다. 파일을 확인한 뒤 직접 정리하세요: {save_dir}")

    model = GPTQModel.load(
        MODEL_ID, quantize_config=quant_config,
        device="cuda:0", dtype=torch.float16
    )
    calibration = None
    if calibration_texts is not None:
        calibration = [
            {
                "input_ids": tokenizer(
                    text,
                    truncation=True,
                    max_length=CALIBRATION_LENGTH
                )["input_ids"]
            }
            for text in calibration_texts
        ]
    model.quantize(calibration, batch_size=1, backend=backend)
    model.save(str(save_dir))
    del model
    clear_gpu_cache()
    return save_dir

def load_quantized(path):
    return GPTQModel.load(
        str(path), device="cuda:0", dtype=torch.float16, backend=BACKEND.GPTQ_TORCH
    )

rtn_path = quantize_and_save(
    "RTN 4bit", RTNConfig(bits=4, group_size=128),
    calibration_texts=None, backend=BACKEND.GPTQ_TORCH
)
rtn_model = load_quantized(rtn_path)
results.extend(evaluate_ppl(rtn_model, eval_token_groups, "RTN 4bit"))
del rtn_model
clear_gpu_cache()
pd.DataFrame(results).tail(6)


INFO  QuantizeConfig: offload_to_disk_path auto set to temporary dir `/tmp/gptqmodel_4tq300il`
기존 체크포인트 사용: /content/drive/MyDrive/Team22/pilot_qwen15b/rtn_4bit
from_quantized: adapter: None
INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]
INFO  The layer model.embed_tokens is not quantized.                            
INFO  Kernel: selected: `TorchLinear`                                           
INFO  Kernel: candidates -> `[TorchLinear]`                                     
INFO  Kernel: per-module selections -> `[TorchLinear=196]`                      
INFO  Kernel: selected -> `TorchLinear`.                                        
INFO  Loader: device = cuda:0                                                   
INFO  Loader: honoring explicit device_map request: {'': 'cuda:0'}              
INFO  Loader: device_map = {'': 'cuda:0'}                                       


INFO  Format: Converting `format` from `FORMAT.GPTQ` to internal `FORMAT.GPTQ_V2`.
INFO  Kernel: selected: `TorchLinear`                                           
INFO  gc.collect() reclaimed 360 objects in 0.450s                              
INFO  Model: Loaded `generation_config`: GenerationConfig {   "bos_token_id": 151643,   "eos_token_id": 151645,   "output_attentions": false,   "output_hidden_states": false,   "pad_token_id": 151643,   "use_cache": true } 
INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.
INFO  Model: Updated `generation_config`: GenerationConfig {   "bos_token_id": 151643,   "do_sample": true,   "eos_token_id": [     151645,     151643   ],   "pad_token_id": 151643,   "repetition_penalty": 1.1,   "temperature": 0.7,   "top_k": 20,   "top_p": 0.8 } 
INFO  Kernel: loaded -> `[TorchLinear]`                                         


,condition,language,group,mean_nll,ppl,predicted_tokens
12,RTN 4bit,ko,1,2.464460,11.757137,8176
13,RTN 4bit,ko,2,2.366479,10.659788,8176
14,RTN 4bit,ko,3,2.609200,13.588171,8176
15,RTN 4bit,en,1,2.705778,14.965953,8176
16,RTN 4bit,en,2,2.812763,16.655871,8176
17,RTN 4bit,en,3,2.852136,17.324752,8176


## 5. 영어·한국어 보정 GPTQ 4bit 생성 및 평가

두 GPTQ 조건은 보정 언어만 다르게 하고 나머지 설정은 같습니다. 각 GPTQ는 FP16 원본에서 새로 시작합니다.


In [39]:
for condition, calibration_texts in [
    ("GPTQ English 4bit", calib_en_texts),
    ("GPTQ Korean 4bit", calib_ko_texts),
]:
    path = quantize_and_save(
        condition, GPTQConfig(bits=4, group_size=128, sym=True, desc_act=False),
        calibration_texts=calibration_texts, backend=BACKEND.GPTQ_TORCH
    )
    quant_model = load_quantized(path)
    results.extend(evaluate_ppl(quant_model, eval_token_groups, condition))
    del quant_model
    clear_gpu_cache()
    print("완료:", condition)


INFO  QuantizeConfig: offload_to_disk_path auto set to temporary dir `/tmp/gptqmodel_rd4iw5zb`


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]
INFO  Loader: using checkpoint-backed lazy turtle source for `/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306`
INFO  Model: Loaded `generation_config`: GenerationConfig {   "bos_token_id": 151643,   "eos_token_id": 151645,   "output_attentions": false,   "output_hidden_states": false,   "pad_token_id": 151643,   "use_cache": true } 
INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.
INFO  Model: Updated `generation_config`: GenerationConfig {   "bos_token_id": 151643,   "do_sample": true,   "eos_token_id": [     151645,     151643   ],   "pad_token_id": 151643,   "repetition_penalty": 1.1,   "temperature": 0.7,   "top_k": 20,   "top_p": 0.8 } 
INFO  Kernel: loaded -> `[]`                                                    
INFO  Packing Kernel: selected: `TorchLinear`        

Writing model shards: 0it [00:00, ?it/s]

INFO  Saved Quantize Config:  {   "bits": 4,   "group_size": 128,   "desc_act": false,   "lm_head": false,   "method": "gptq",   "quant_method": "gptq",   "format": "gptq",   "checkpoint_format": "gptq",   "pack_dtype": "int32",   "meta": {     "quantizer": [       "gptqmodel:7.10.0"     ],     "uri": "https://github.com/modelcloud/gptqmodel",     "damp_percent": 0.05,     "damp_auto_increment": 0.01,     "static_groups": false,     "true_sequential": true,     "mse": 0.0,     "gptaq": null,     "foem": null,     "act_group_aware": true,     "fallback": {       "strategy": "rtn",       "threshold": "0.5%",       "smooth": null     },     "offload_to_disk": true,     "telemetry": {       "device": false     },     "offload_to_disk_path": "/tmp/gptqmodel_rd4iw5zb",     "pack_impl": "auto",     "gc_mode": "interval",     "wait_for_submodule_finalizers": false,     "auto_forward_data_parallel": true,     "dense_vram_strategy": "exclusive",     "dense_vram_strategy_devices": null,     "moe_

... (3 outputs left out)

INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]
INFO  Loader: using checkpoint-backed lazy turtle source for `/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306`
INFO  Model: Loaded `generation_config`: GenerationConfig {   "bos_token_id": 151643,   "eos_token_id": 151645,   "output_attentions": false,   "output_hidden_states": false,   "pad_token_id": 151643,   "use_cache": true } 
INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.
INFO  Model: Updated `generation_config`: GenerationConfig {   "bos_token_id": 151643,   "do_sample": true,   "eos_token_id": [     151645,     151643   ],   "pad_token_id": 151643,   "repetition_penalty": 1.1,   "temperature": 0.7,   "top_k": 20,   "top_p": 0.8 } 
INFO  Kernel: loaded -> `[]`                                                    
INFO  Packing Kernel: selected: `TorchLinear`        

Writing model shards: 0it [00:00, ?it/s]

INFO  Saved Quantize Config:  {   "bits": 4,   "group_size": 128,   "desc_act": false,   "lm_head": false,   "method": "gptq",   "quant_method": "gptq",   "format": "gptq",   "checkpoint_format": "gptq",   "pack_dtype": "int32",   "meta": {     "quantizer": [       "gptqmodel:7.10.0"     ],     "uri": "https://github.com/modelcloud/gptqmodel",     "damp_percent": 0.05,     "damp_auto_increment": 0.01,     "static_groups": false,     "true_sequential": true,     "mse": 0.0,     "gptaq": null,     "foem": null,     "act_group_aware": true,     "fallback": {       "strategy": "rtn",       "threshold": "0.5%",       "smooth": null     },     "offload_to_disk": true,     "telemetry": {       "device": false     },     "offload_to_disk_path": "/tmp/gptqmodel_e4hetoeq",     "pack_impl": "auto",     "gc_mode": "interval",     "wait_for_submodule_finalizers": false,     "auto_forward_data_parallel": true,     "dense_vram_strategy": "exclusive",     "dense_vram_strategy_devices": null,     "moe_

INFO  Format: Converting `format` from `FORMAT.GPTQ` to internal `FORMAT.GPTQ_V2`.
INFO  Kernel: selected: `TorchLinear`                                           
INFO  gc.collect() reclaimed 384 objects in 0.436s                              
INFO  Model: Loaded `generation_config`: GenerationConfig {   "bos_token_id": 151643,   "eos_token_id": 151645,   "output_attentions": false,   "output_hidden_states": false,   "pad_token_id": 151643,   "use_cache": true } 
INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.
INFO  Model: Updated `generation_config`: GenerationConfig {   "bos_token_id": 151643,   "do_sample": true,   "eos_token_id": [     151645,     151643   ],   "pad_token_id": 151643,   "repetition_penalty": 1.1,   "temperature": 0.7,   "top_k": 20,   "top_p": 0.8 } 
INFO  Kernel: loaded -> `[TorchLinear]`                                         
완료: GPTQ Korean 4bit


## 6. 표와 예비 결과 정리

아래 표에서 `D`는 언어별 FP16 기준 Perplexity 증가율입니다. `D_gap_pp`가 양수면 해당 평가 묶음에서 한국어 저하율이 영어보다 높았다는 뜻입니다. 이는 예비 결과를 요약하는 값이며, 지금 여기서 1번/3번 선택을 자동으로 결정하지 않습니다.


In [44]:
for file in OUTPUT_DIR.glob("*.csv"):
    print(file.resolve())

/content/drive/MyDrive/Team22/pilot_qwen15b/korean_calibration_effect.csv
/content/drive/MyDrive/Team22/pilot_qwen15b/language_gap_summary.csv
/content/drive/MyDrive/Team22/pilot_qwen15b/ppl_by_group.csv
/content/drive/MyDrive/Team22/pilot_qwen15b/ppl_summary.csv


In [41]:
# 1. 결과 데이터프레임 생성
df = pd.DataFrame(results)

# 2. 중복 결과 확인
key_cols = ["condition", "language", "group"]

duplicates = df[df.duplicated(subset=key_cols, keep=False)]

print(f"전체 결과: {len(df)}개")
print(f"중복 키에 해당하는 행: {len(duplicates)}개")

if not duplicates.empty:
    display(
        duplicates.sort_values(key_cols)[
            key_cols + ["ppl"]
        ]
    )

# 3. 동일한 평가 조건의 중복 결과 제거
# 마지막으로 기록된 결과를 유지
df = df.drop_duplicates(
    subset=key_cols,
    keep="last"
).copy()

print(f"중복 제거 후: {len(df)}개")

# 4. FP16 기준 PPL 설정
base = (
    df[df["condition"] == "FP16"]
    .set_index(["language", "group"])["ppl"]
)

# 5. FP16 대비 PPL 변화율 계산
df["D_percent"] = df.apply(
    lambda row: (
        (
            row["ppl"]
            / base.loc[(row["language"], row["group"])]
            - 1
        ) * 100
        if row["condition"] != "FP16"
        else 0.0
    ),
    axis=1
)

# 6. 한국어-영어 성능 저하율 차이
wide_d = df.pivot_table(
    index=["condition", "group"],
    columns="language",
    values="D_percent"
).reset_index()

wide_d["D_gap_pp"] = wide_d["ko"] - wide_d["en"]

# 7. 언어별 평균 결과
summary = (
    df.groupby(["condition", "language"], as_index=False)
    .agg(
        mean_ppl=("ppl", "mean"),
        sd_ppl=("ppl", "std"),
        mean_D_percent=("D_percent", "mean")
    )
)

# 8. 언어 간 성능 저하율 차이 요약
gap_summary = (
    wide_d.groupby("condition", as_index=False)
    .agg(
        mean_D_gap_pp=("D_gap_pp", "mean"),
        positive_gap_groups=(
            "D_gap_pp",
            lambda x: int((x > 0).sum())
        ),
        n_groups=("D_gap_pp", "count")
    )
)

# 9. 영어 보정 GPTQ vs 한국어 보정 GPTQ
ko_compare = df[df["language"] == "ko"].pivot(
    index="group",
    columns="condition",
    values="ppl"
)

ko_calibration_benefit = (
    (
        ko_compare["GPTQ English 4bit"]
        - ko_compare["GPTQ Korean 4bit"]
    )
    / ko_compare["GPTQ English 4bit"]
    * 100
).rename("Korean_calibration_PPL_improvement_percent")

# 10. 결과 출력
print("언어별 평균 결과")
display(summary)

print("한국어-영어 저하율 차이 (percentage points)")
display(gap_summary)

print("한국어 보정 GPTQ의 한국어 PPL 개선율")
display(ko_calibration_benefit.to_frame())

# 11. CSV 저장
df.to_csv(
    OUTPUT_DIR / "ppl_by_group.csv",
    index=False
)

summary.to_csv(
    OUTPUT_DIR / "ppl_summary.csv",
    index=False
)

gap_summary.to_csv(
    OUTPUT_DIR / "language_gap_summary.csv",
    index=False
)

ko_calibration_benefit.to_csv(
    OUTPUT_DIR / "korean_calibration_effect.csv"
)

print("모든 분석 결과 저장 완료")
if not DRIVE_AVAILABLE:
    from zipfile import ZipFile
    archive = OUTPUT_DIR / "pilot_results.zip"
    with ZipFile(archive, "w") as bundle:
        for name in ("runtime.json", "pilot_samples.json", "ppl_by_group.csv",
                     "ppl_summary.csv", "language_gap_summary.csv",
                     "korean_calibration_effect.csv"):
            bundle.write(OUTPUT_DIR / name, arcname=name)
    files.download(str(archive))
    print("다운로드 파일:", archive.name)


전체 결과: 30개
중복 키에 해당하는 행: 12개


,condition,language,group,ppl
9,RTN 4bit,en,1,14.965953
15,RTN 4bit,en,1,14.965953
10,RTN 4bit,en,2,16.655871
16,RTN 4bit,en,2,16.655871
11,RTN 4bit,en,3,17.324752
17,RTN 4bit,en,3,17.324752
6,RTN 4bit,ko,1,11.757137
12,RTN 4bit,ko,1,11.757137
7,RTN 4bit,ko,2,10.659788
13,RTN 4bit,ko,2,10.659788


중복 제거 후: 24개
언어별 평균 결과


,condition,language,mean_ppl,sd_ppl,mean_D_percent
0,FP16,en,13.789369,0.847626,0.000000
1,FP16,ko,9.553037,1.243826,0.000000
2,GPTQ English 4bit,en,14.903912,1.015329,8.052798
3,GPTQ English 4bit,ko,11.617115,1.362222,21.749348
4,GPTQ Korean 4bit,en,15.475669,1.001349,12.215884
5,GPTQ Korean 4bit,ko,10.146582,1.314187,6.222088
6,RTN 4bit,en,16.315525,1.215672,18.256736
7,RTN 4bit,ko,12.001699,1.479430,25.721641


한국어-영어 저하율 차이 (percentage points)


,condition,mean_D_gap_pp,positive_gap_groups,n_groups
0,FP16,0.000000,0,3
1,GPTQ English 4bit,13.696550,3,3
2,GPTQ Korean 4bit,-5.993795,0,3
3,RTN 4bit,7.464905,3,3


한국어 보정 GPTQ의 한국어 PPL 개선율


,Korean_calibration_PPL_improvement_percent
group,
1,12.503123
2,13.966158
3,11.758081


모든 분석 결과 저장 완료


In [43]:
print("저장 위치:", OUTPUT_DIR.resolve())

저장 위치: /content/drive/MyDrive/Team22/pilot_qwen15b


In [40]:
df = pd.DataFrame(results)
base = (df[df["condition"] == "FP16"]
        .set_index(["language", "group"])["ppl"])
df["D_percent"] = df.apply(
    lambda row: (row["ppl"] / base.loc[(row["language"], row["group"])] - 1) * 100
    if row["condition"] != "FP16" else 0.0, axis=1
)

wide_d = df.pivot_table(
    index=["condition", "group"], columns="language", values="D_percent"
).reset_index()
wide_d["D_gap_pp"] = wide_d["ko"] - wide_d["en"]

summary = (df.groupby(["condition", "language"], as_index=False)
           .agg(mean_ppl=("ppl", "mean"), sd_ppl=("ppl", "std"),
                mean_D_percent=("D_percent", "mean")))
gap_summary = (wide_d.groupby("condition", as_index=False)
               .agg(mean_D_gap_pp=("D_gap_pp", "mean"),
                    positive_gap_groups=("D_gap_pp", lambda x: int((x > 0).sum())),
                    n_groups=("D_gap_pp", "count")))

# 한국어 보정 GPTQ가 영어 보정 GPTQ보다 한국어 PPL을 낮췄는지 요약합니다.
ko_compare = df[df["language"] == "ko"].pivot(
    index="group", columns="condition", values="ppl"
)
ko_calibration_benefit = (
    (ko_compare["GPTQ English 4bit"] - ko_compare["GPTQ Korean 4bit"])
    / ko_compare["GPTQ English 4bit"] * 100
).rename("Korean_calibration_PPL_improvement_percent")

print("언어별 평균 결과")
display(summary)
print("한국어-영어 저하율 차이 (percentage points)")
display(gap_summary)
print("한국어 평가에서 한국어 보정이 영어 보정보다 개선한 비율 (양수면 개선)")
display(ko_calibration_benefit.to_frame())

df.to_csv(OUTPUT_DIR / "ppl_by_group.csv", index=False)
summary.to_csv(OUTPUT_DIR / "ppl_summary.csv", index=False)
gap_summary.to_csv(OUTPUT_DIR / "language_gap_summary.csv", index=False)
ko_calibration_benefit.to_csv(OUTPUT_DIR / "korean_calibration_effect.csv")


ValueError: Index contains duplicate entries, cannot reshape

## 7. 결과를 보고 확인할 것

- **언어별 양자화 영향:** RTN/GPTQ마다 `D_percent`와 `D_gap_pp`를 확인합니다. 세 평가 묶음에서 방향이 같은지 함께 봅니다.
- **보정 언어 영향:** 한국어 평가에서 `GPTQ Korean 4bit`의 PPL이 `GPTQ English 4bit`보다 낮은지 봅니다. 반대 언어 평가도 같이 확인합니다.
- **변동성:** 묶음별 PPL과 저하율을 나란히 보고, 평균만으로 결론 내리지 않습니다.
- **주제 선택:** 결과를 팀에서 검토한 뒤 1번과 3번 중 하나를 고릅니다. 3번을 택하면 그때 부품 민감도와 혼합 정밀도 후보를 설계합니다.
- **해석 한계:** 서로 다른 말뭉치의 절대 PPL을 한국어-영어 간 직접 비교하지 않습니다. 이 예비 실험은 퍼플렉시티 변화의 방향을 보는 것이며, KoBEST 정확도나 실제 대화 품질을 대신하지 않습니다.

결과 파일은 `/content/team22_pilot`에 저장됩니다. Colab 세션이 끝나기 전에 폴더를 Google Drive로 복사하거나 내려받으세요.
